# PTCG Merged Agent Workbench

Parent notebook for **The Pokémon Company - PTCG AI Battle Challenge Simulation**.

This workbench merges three public reference threads into one submission pipeline:

| Source | Notebook | What we take |
|--------|----------|--------------|
| **Dragapult agent** | `docs/resources/reference_notebooks/a-sample-rule-based-agent-dragapult-ex-deck.ipynb` | Full policy skeleton: log tracking, deck reconstruction, combo planning, contextual scoring |
| **Expectimax agent** | `docs/resources/reference_notebooks/improved-probabilistic-agent.ipynb` | Search API integration with UCB1 + opponent archetype detection |
| **Meta snapshot author** | `docs/resources/reference_notebooks/pok-mon-tcg-ai-battle-meta-snapshot-07-july.ipynb` | Meta-informed deck choice + holdout validation mindset (agent code is base64-only) |

## Merge thesis

Each public notebook has one blind spot the others fill:

- **Dragapult** → strong policy, **no search**
- **Expectimax** → real lookahead, **weaker base policy**
- **Meta author** → ladder analysis, **hidden agent implementation**

**Target architecture:** Dragapult policy skeleton + Expectimax search/opponent reads + meta-informed deck under `data/`.

> Put your chosen deck CSV in `data/deck.csv` before running the build cells below.


## 1. Environment setup

Paths assume this notebook lives in `notebooks/`. Competition SDK and deck files belong in `data/` at repo root.


In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
DATA_DIR = REPO_ROOT / "data"
DOCS_RESOURCES = REPO_ROOT / "docs" / "resources" / "reference_notebooks"
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"

for path in (DATA_DIR, NOTEBOOKS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Repo:", REPO_ROOT)
print("Data:", DATA_DIR)
print("Reference notebooks:", DOCS_RESOURCES)
print("Deck present:", (DATA_DIR / "deck.csv").exists())


## 2. Meta-informed deck choice (from Meta Snapshot)

The meta snapshot author tracks ladder share and conversion. Their agent logic is **not public** (`main_b64` / `deck_b64`), but the **decision framework** is:

1. Prefer an underexplored archetype with solid conversion (e.g. **Starmie**, **Festival Thwackey**) instead of copying the highest-share deck blindly.
2. Stress-test against mandatory ladder pillars: **Starmie**, **Hop/Trevenant**, **Archaludon**, **Lucario**.
3. Promote builds only after **holdout** gates pass — not from live ladder noise alone.

Use the table below as a deck-selection checklist before you lock `data/deck.csv`.


In [ ]:
import pandas as pd

META_FIELD = pd.DataFrame([
    {"archetype": "alakazam_dunsparce", "usage_pct": 18.96, "score_pct": 51.34, "notes": "High share, coin-flippy"},
    {"archetype": "lucario", "usage_pct": 18.78, "score_pct": 42.44, "notes": "Crowded — Expectimax baseline lives here"},
    {"archetype": "hop_trevenant", "usage_pct": 17.52, "score_pct": 45.47, "notes": "Mandatory stress test"},
    {"archetype": "archaludon", "usage_pct": 14.57, "score_pct": 62.20, "notes": "Top converter, heavily targeted"},
    {"archetype": "starmie", "usage_pct": 13.85, "score_pct": 51.89, "notes": "Tempo stress test + underexplored candidate"},
    {"archetype": "dragapult", "usage_pct": 7.31, "score_pct": 49.13, "notes": "Policy skeleton source in this repo"},
    {"archetype": "festival_thwackey", "usage_pct": 1.16, "score_pct": 45.99, "notes": "Low share — meta-informed dark horse"},
])

META_FIELD.sort_values("usage_pct", ascending=False)[["archetype", "usage_pct", "score_pct", "notes"]]


### Deck selection workflow

1. Export your target list from Kaggle competition data into `data/deck.csv` (60 lines, one card id per line).
2. Update card constants in the merged agent if you switch archetype away from Dragapult.
3. Re-run the build + holdout cells before submitting.


In [ ]:
DECK_PATH = DATA_DIR / "deck.csv"
if DECK_PATH.exists():
    deck = [int(line) for line in DECK_PATH.read_text().splitlines() if line.strip()]
    assert len(deck) == 60, f"Expected 60 cards, got {len(deck)}"
    print("Loaded deck.csv:", len(deck), "cards,", len(set(deck)), "unique ids")
else:
    print("No deck yet — add data/deck.csv before building a submission.")


## 3. Extract map — what came from where

### From Dragapult (`a-sample-rule-based-agent-dragapult-ex-deck.ipynb`)

- `AttackPlan`, `pre_turn_log` / `current_turn_log`
- `set_card_counts()` deck reconstruction + prize inference
- `main_option_proc()` Phantom Dive combo planning
- Contextual option scoring loop → `DragapultPolicy.choose()`

### From Expectimax (`improved-probabilistic-agent.ipynb`)

- `opponent_is_water_deck()` / `opponent_is_crustle_wall()`
- `evaluate_state()`, `simulate_action()`, `rollout_turn()`
- `SEARCH_ALGO()` with exploration + **UCB1** (wired to `DragapultPolicy`)

### From Meta Snapshot (`pok-mon-tcg-ai-battle-meta-snapshot-07-july.ipynb`)

- Field usage / conversion framing (section 2)
- Holdout gate scaffold (section 5)
- **Not imported:** base64 agent payloads


## 4. Build merged `main.py`

Run the builder to re-sync from the three reference notebooks, then copy the result to repo root.

Source references:
- `docs/resources/reference_notebooks/a-sample-rule-based-agent-dragapult-ex-deck.ipynb`
- `docs/resources/reference_notebooks/improved-probabilistic-agent.ipynb`


In [ ]:
import subprocess
import sys
from pathlib import Path

builder = NOTEBOOKS_DIR / "build_merged_agent.py"
subprocess.run([sys.executable, str(builder)], check=True)

merged_template = NOTEBOOKS_DIR / "merged_agent_main.py"
main_src = merged_template.read_text()
(REPO_ROOT / "main.py").write_text(main_src)
print("Wrote", REPO_ROOT / "main.py", f"({len(main_src.splitlines())} lines)")


## 5. Holdout validation scaffold

Mirror the meta author's gate: benchmark locally against a fixed opponent pool before burning daily Kaggle submissions.

Fill in `HOLDOUT_OPPONENTS` once you have baseline agents or recorded replays. The function below is a stub you can wire to `kaggle-environments` or the cabt SDK.


In [ ]:
HOLDOUT_OPPONENTS = [
    "starmie",
    "hop_trevenant",
    "archaludon",
    "lucario",
    "dragapult",
]

HOLDOUT_GAMES = 40
PROMOTE_THRESHOLD = 0.52  # win rate needed to consider submission-ready


def run_holdout_suite(main_module="main", opponents=HOLDOUT_OPPONENTS, games=HOLDOUT_GAMES):
    """Stub: plug in local simulator calls here.

    Expected return shape:
        [{"opponent": str, "wins": int, "losses": int, "ties": int}, ...]
    """
    raise NotImplementedError(
        "Connect this to kaggle-environments / cabt once data/cg is available."
    )


def summarize_holdout(results):
    rows = []
    for row in results:
        total = row["wins"] + row["losses"] + row["ties"]
        rate = row["wins"] / total if total else 0.0
        verdict = "PROMOTE_CANDIDATE" if rate >= PROMOTE_THRESHOLD else "HOLD_DO_NOT_SUBMIT"
        rows.append({**row, "win_rate": rate, "verdict": verdict})
    return pd.DataFrame(rows)


print("Holdout opponents:", HOLDOUT_OPPONENTS)
print("Promotion threshold:", PROMOTE_THRESHOLD)


## 6. Package Kaggle submission

Submission requires `main.py`, `deck.csv`, and the `cg/` SDK at the **top level** of the tarball.


In [ ]:
import glob
import shutil
import tarfile
from pathlib import Path


def find_cg_dir():
    patterns = [
        str(DATA_DIR / "cg"),
        str(REPO_ROOT / "cg"),
        "/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg",
        "/kaggle/input/**/sample_submission/cg",
        "/kaggle/input/**/cg-lib/cg",
    ]
    for pattern in patterns:
        for path in glob.glob(pattern, recursive=True):
            if Path(path).is_dir() and (Path(path) / "api.py").exists():
                return Path(path)
    raise FileNotFoundError("Place the cg SDK under data/cg before packaging.")


def build_submission(output=REPO_ROOT / "submission.tar.gz"):
    cg_src = find_cg_dir()
    staging = REPO_ROOT / ".submission_staging"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir()

    shutil.copy2(REPO_ROOT / "main.py", staging / "main.py")
    shutil.copy2(DECK_PATH, staging / "deck.csv")
    shutil.copytree(cg_src, staging / "cg")

    with tarfile.open(output, "w:gz") as tar:
        tar.add(staging / "main.py", arcname="main.py")
        tar.add(staging / "deck.csv", arcname="deck.csv")
        for item in sorted((staging / "cg").rglob("*")):
            if item.is_file() and "__pycache__" not in item.parts:
                tar.add(item, arcname=str(Path("cg") / item.relative_to(staging / "cg")))

    shutil.rmtree(staging)
    print("Created:", output, f"({output.stat().st_size / 1024 / 1024:.2f} MiB)")


# build_submission()  # uncomment after data/deck.csv and data/cg exist


## 7. Next steps

1. Drop competition files into `data/` (`deck.csv`, `cg/`).
2. Run sections 2 → 4 → 5 (holdout) → 6 (submission).
3. Iterate policy constants if you pivot from Dragapult skeleton to Starmie/Festival lists.
4. Keep reference notebooks in `docs/resources/` (gitignored) for diffing against public baselines.
